# Project Fertility PSP v6.2 Final Report

## 난임 환자 대상 임신 성공 여부 예측 모델링

본 노트북은 난임 시술 데이터를 바탕으로 임신 성공 여부를 예측하고, 제출 형식에 맞는 `submission.csv`를 생성하기 위한 최종 실행본이다.  
분석 흐름은 데이터 점검, 임상적 의미를 고려한 Feature Engineering, OOF 기반 검증, 앙상블, 제출 파일 검증 순서로 구성하였다.

## 목차

1. 실행 환경 및 Config
2. 데이터 로드와 기본 검증
3. 임상/문헌 기반 변수 그룹 정의
4. Feature Engineering
5. 전처리 파이프라인
6. OOF 검증과 모델 학습
7. Prediction Bank와 앙상블
8. 결과 해석 및 제출용 요약
9. Submission 생성
10. 참고문헌

## 분석 원칙

- 입력 데이터는 `train.csv`, `test.csv`, `sample_submission.csv`만 사용한다.
- 예측 대상인 `임신 성공 여부`는 학습 전에 0/1 라벨로 정리하고, 사용할 수 없는 라벨은 학습에서 제외한다.
- Feature Engineering은 각 행에 이미 기록된 시술 정보만 사용하며, target을 이용해 값을 채우지 않는다.
- 제출 파일은 `sample_submission.csv`의 행 수, 컬럼 순서, ID 순서를 그대로 유지한다.
- 모델 결과는 실제 의학적 판단을 대체하지 않으며, 제공된 데이터 안에서의 예측 신호로 해석한다.


## 1. 실행 환경 및 Config

실행 모드는 빠른 점검용 `smoke`와 최종 실행용 `full`로 나누었다.  
산출물은 기본적으로 `outputs/team_report_v6_3/`에 저장하며, 제출 파일명은 `submission_team_report_v6_3_...csv` 형식으로 생성한다.


In [ ]:
from __future__ import annotations

import json
import os
import random
import warnings
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

warnings.filterwarnings("ignore")

RUN_MODE = os.environ.get("PSP_RUN_MODE", "smoke").strip().lower()
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError(f"PSP_RUN_MODE must be 'smoke' or 'full'. Got: {RUN_MODE!r}")

DEFAULT_OUTPUT_DIR = "outputs/team_report_v6_3_smoke" if RUN_MODE == "smoke" else "outputs/team_report_v6_3"

@dataclass(frozen=True)
class Config:
    seed: int = int(os.environ.get("PSP_SEED", "42"))
    run_mode: str = RUN_MODE
    n_splits: int = int(os.environ.get("PSP_N_SPLITS", "2" if RUN_MODE == "smoke" else "5"))
    data_dir: Path = Path(os.environ.get("PSP_DATA_DIR", "data" if Path("data").exists() else "."))
    train_path: Path = Path(os.environ.get("PSP_TRAIN_FILE", "train.csv"))
    test_path: Path = Path(os.environ.get("PSP_TEST_FILE", "test.csv"))
    sample_submission_path: Path = Path(os.environ.get("PSP_SAMPLE_SUBMISSION_FILE", "sample_submission.csv"))
    output_dir: Path = Path(os.environ.get("PSP_OUTPUT_DIR", DEFAULT_OUTPUT_DIR))
    target_candidates: tuple[str, ...] = (
        "임신 성공 여부",
        "pregnancy_success",
        "target",
        "label",
        "y",
    )
    id_candidates: tuple[str, ...] = (
        "ID",
        "id",
        "시술 ID",
        "환자 ID",
    )
    submission_prefix: str = "submission_team_report_v6_3"

CFG = Config()
CFG.output_dir.mkdir(parents=True, exist_ok=True)

random.seed(CFG.seed)
np.random.seed(CFG.seed)

print(f"Run mode: {CFG.run_mode}")
print(f"Data directory: {CFG.data_dir}")
print(f"Output directory: {CFG.output_dir}")
print(f"Requested CV splits: {CFG.n_splits}")


## 2. 데이터 로드와 기본 검증

본 노트북은 외부 중간 산출물이나 이전 제출 파일을 사용하지 않는다. 입력은 `train.csv`, `test.csv`, `sample_submission.csv` 세 파일로 제한한다.


In [ ]:
def resolve_path(base_dir: Path, path: Path) -> Path:
    candidate = path if path.is_absolute() else base_dir / path
    if not candidate.exists():
        raise FileNotFoundError(f"Required input file not found: {candidate}")
    return candidate


def find_first_existing(columns: pd.Index, candidates: tuple[str, ...]) -> str | None:
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None


train = pd.read_csv(resolve_path(CFG.data_dir, CFG.train_path))
test = pd.read_csv(resolve_path(CFG.data_dir, CFG.test_path))
sample_submission = pd.read_csv(resolve_path(CFG.data_dir, CFG.sample_submission_path))

target_col = find_first_existing(train.columns, CFG.target_candidates)
if target_col is None:
    raise ValueError(f"Target column not found. Candidates: {CFG.target_candidates}")

id_col = find_first_existing(test.columns, CFG.id_candidates)

print("train shape:", train.shape)
print("test shape:", test.shape)
print("sample_submission shape:", sample_submission.shape)
print("target column:", target_col)
print("id column:", id_col)


## 3. 임상/문헌 기반 변수 그룹 정의

변수 그룹은 모델 입력을 임상적으로 점검하기 위한 보조 구조이다. 여성 연령, 난임 원인, 시술 유형, 배아/난자/이식 관련 변수는 IVF/ICSI 및 FET outcome 해석에서 반복적으로 논의되는 축이므로 결과 해석 시 과도한 인과 해석을 피하고 예측 변수로서의 의미를 중심으로 기술한다 (Murugappan et al., 2023; Carson & Kallen, 2021; Liu et al., 2021).


In [ ]:
CLINICAL_KEYWORDS = {
    "age": ["나이", "연령", "age"],
    "infertility": ["난임", "불임", "infertility", "원인"],
    "procedure": ["시술", "IVF", "ICSI", "IUI", "FET", "FER", "해동", "동결"],
    "embryo": ["배아", "embryo", "blastocyst", "포배"],
    "oocyte": ["난자", "oocyte", "egg"],
    "sperm": ["정자", "sperm", "male"],
    "transfer": ["이식", "transfer"],
}


def match_columns_by_keywords(columns: pd.Index, keywords: list[str]) -> list[str]:
    lowered = {col: str(col).lower() for col in columns}
    return [
        col for col, low in lowered.items()
        if any(keyword.lower() in low for keyword in keywords)
    ]


feature_columns = [col for col in train.columns if col != target_col]
if id_col in feature_columns:
    feature_columns.remove(id_col)

clinical_groups = {
    group: match_columns_by_keywords(pd.Index(feature_columns), keywords)
    for group, keywords in CLINICAL_KEYWORDS.items()
}

for group, cols in clinical_groups.items():
    print(f"{group:>12}: {len(cols)} columns")
    print(cols[:10])


## 4. Feature Engineering

초기 EDA에서 중요하게 보았던 결측 구조, 나이 구간, 배아·난자 수량, 시술 유형, 기증/ICSI/FET 관련 단서를 모델 입력으로 정리한다.  
특히 배아·난자·이식 관련 결측은 단순 누락이 아니라 시술 유형과 기록 방식의 차이를 반영할 수 있으므로, 결측 여부를 별도 feature로 보존한다.

이 단계에서 추가하는 주요 변수는 다음과 같다.

- 배아·난자·이식 관련 결측 count/rate 및 missing flag
- 나이 순서형 변수와 고연령 구간 flag
- 난자·배아·이식 수량 구간화 변수
- FER/FET, ICSI, 자연주기 가능성 flag
- 나이×난자 출처, 시술 유형×세부 시술 등 조합 변수
- 생성 배아 대비 이식/저장 비율 등 주요 ratio 변수
- train 기준 IQR clipping 후보 변수

모든 규칙은 train과 test에 동일하게 적용하며, clipping 경계값은 train 데이터에서만 계산한다.


In [ ]:
def add_missing_pattern_features(df: pd.DataFrame, reference_columns: list[str]) -> pd.DataFrame:
    out = df.copy()

    embryo_cols = match_columns_by_keywords(pd.Index(reference_columns), CLINICAL_KEYWORDS["embryo"])
    oocyte_cols = match_columns_by_keywords(pd.Index(reference_columns), CLINICAL_KEYWORDS["oocyte"])
    transfer_cols = match_columns_by_keywords(pd.Index(reference_columns), CLINICAL_KEYWORDS["transfer"])
    procedure_cols = match_columns_by_keywords(pd.Index(reference_columns), CLINICAL_KEYWORDS["procedure"])

    grouped_missing_specs = {
        "f_embryo_missing_count": embryo_cols,
        "f_oocyte_missing_count": oocyte_cols,
        "f_transfer_missing_count": transfer_cols,
        "f_procedure_missing_count": procedure_cols,
    }

    for feature_name, cols in grouped_missing_specs.items():
        available = [col for col in cols if col in out.columns]
        out[feature_name] = out[available].isna().sum(axis=1) if available else 0

    embryo_available = [col for col in embryo_cols if col in out.columns]
    out["f_embryo_all_missing"] = (
        out[embryo_available].isna().all(axis=1).astype(int)
        if embryo_available
        else 0
    )

    reference_available = [col for col in reference_columns if col in out.columns]
    out["f_any_clinical_missing_count"] = (
        out[reference_available].isna().sum(axis=1)
        if reference_available
        else 0
    )
    out["f_any_clinical_missing_ratio"] = (
        out["f_any_clinical_missing_count"] / max(len(reference_available), 1)
    )

    return out


def to_numeric_safe_series(df: pd.DataFrame, col: str) -> pd.Series:
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce")
    return pd.Series(np.nan, index=df.index)


def first_existing(columns: pd.Index | list[str], candidates: list[str]) -> str | None:
    column_set = set(columns)
    for candidate in candidates:
        if candidate in column_set:
            return candidate
    return None


def safe_text_series(df: pd.DataFrame, col: str | None) -> pd.Series:
    if col is None or col not in df.columns:
        return pd.Series("missing", index=df.index, dtype="string")
    return df[col].astype("string").fillna("missing")


def safe_ratio(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    num = pd.to_numeric(numerator, errors="coerce")
    den = pd.to_numeric(denominator, errors="coerce").replace(0, np.nan)
    result = num / den
    return result.replace([np.inf, -np.inf], np.nan)


AGE_ORD_MAP = {
    "만18-34세": 0,
    "만35-37세": 1,
    "만38-39세": 2,
    "만40-42세": 3,
    "만43-44세": 4,
    "만45-50세": 5,
}


def make_oocyte_bin(series: pd.Series) -> pd.Series:
    x = pd.to_numeric(series, errors="coerce")
    result = pd.Series("missing", index=series.index, dtype="object")
    result[x == 0] = "0개"
    result[(x >= 1) & (x <= 5)] = "1-5개"
    result[(x >= 6) & (x <= 10)] = "6-10개"
    result[x >= 11] = "11개 이상"
    return result


def make_embryo_bin(series: pd.Series) -> pd.Series:
    x = pd.to_numeric(series, errors="coerce")
    result = pd.Series("missing", index=series.index, dtype="object")
    result[x == 0] = "0개"
    result[(x >= 1) & (x <= 2)] = "1-2개"
    result[(x >= 3) & (x <= 5)] = "3-5개"
    result[x >= 6] = "6개 이상"
    return result


def make_transfer_embryo_bin(series: pd.Series) -> pd.Series:
    x = pd.to_numeric(series, errors="coerce")
    result = pd.Series("missing", index=series.index, dtype="object")
    result[x == 0] = "0개"
    result[(x >= 1) & (x <= 2)] = "1-2개"
    result[x >= 3] = "3개 이상"
    return result


def add_v5_style_domain_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # 1) Structural missing flags from embryo/oocyte/transfer process variables.
    process_missing_cols = [
        "배아 생성 주요 이유",
        "단일 배아 이식 여부",
        "착상 전 유전 진단 사용 여부",
        "총 생성 배아 수",
        "미세주입된 난자 수",
        "미세주입에서 생성된 배아 수",
        "이식된 배아 수",
        "미세주입 배아 이식 수",
        "저장된 배아 수",
        "미세주입 후 저장된 배아 수",
        "해동된 배아 수",
        "해동 난자 수",
        "수집된 신선 난자 수",
        "저장된 신선 난자 수",
        "혼합된 난자 수",
        "파트너 정자와 혼합된 난자 수",
        "기증자 정자와 혼합된 난자 수",
        "동결 배아 사용 여부",
        "신선 배아 사용 여부",
    ]
    available_process_cols = [col for col in process_missing_cols if col in out.columns]
    process_flag_cols = []
    for col in available_process_cols:
        flag_col = f"{col}_missing_flag"
        out[flag_col] = out[col].isna().astype(int)
        process_flag_cols.append(flag_col)

    out["process_missing_count"] = out[process_flag_cols].sum(axis=1) if process_flag_cols else 0
    out["process_missing_rate"] = out["process_missing_count"] / max(len(process_flag_cols), 1)

    # 2) General missing flags for other columns.
    base_cols = list(out.columns)
    for col in base_cols:
        if col not in available_process_cols and out[col].isna().any():
            out[f"{col}_is_missing"] = out[col].isna().astype(int)

    # 3) Clinical inference flags.
    specific_proc_col = first_existing(out.columns, ["특정 시술 유형", "specific_procedure"])
    specific_proc = safe_text_series(out, specific_proc_col)

    out["fer_inferred_flag"] = (
        to_numeric_safe_series(out, "해동된 배아 수").fillna(0) >= 1
    ).astype(int)

    out["icsi_inferred_flag"] = (
        to_numeric_safe_series(out, "미세주입된 난자 수").fillna(0) >= 1
    ).astype(int)

    ovulation_col = first_existing(out.columns, ["배란 자극 여부", "배란 유도 유형"])
    if ovulation_col is not None:
        ov = out[ovulation_col]
        ov_num = pd.to_numeric(ov, errors="coerce")
        ov_text = ov.astype("string").str.lower().fillna("")
        out["natural_cycle_inferred_flag"] = (
            (ov_num == 0) |
            ov_text.str.contains("자연|natural", regex=True)
        ).astype(int)
    else:
        out["natural_cycle_inferred_flag"] = 0

    out["unknown_proc_with_thawed_embryo_flag"] = (
        specific_proc.isin(["Unknown", "알 수 없음", "missing"])
        & (to_numeric_safe_series(out, "해동된 배아 수").fillna(0) >= 1)
    ).astype(int)

    out["unknown_proc_with_icsi_oocyte_flag"] = (
        specific_proc.isin(["Unknown", "알 수 없음", "missing"])
        & (to_numeric_safe_series(out, "미세주입된 난자 수").fillna(0) >= 1)
    ).astype(int)

    # 4) Age ordering and clinically useful age flags.
    age_col = first_existing(out.columns, ["시술 당시 나이", "나이", "age"])
    age_text = safe_text_series(out, age_col)
    out["age_ord"] = age_text.map(AGE_ORD_MAP).astype("float")
    if out["age_ord"].isna().all() and age_col is not None:
        age_num = pd.to_numeric(out[age_col], errors="coerce")
        out.loc[age_num < 35, "age_ord"] = 0
        out.loc[(age_num >= 35) & (age_num <= 37), "age_ord"] = 1
        out.loc[(age_num >= 38) & (age_num <= 39), "age_ord"] = 2
        out.loc[(age_num >= 40) & (age_num <= 42), "age_ord"] = 3
        out.loc[(age_num >= 43) & (age_num <= 44), "age_ord"] = 4
        out.loc[age_num >= 45, "age_ord"] = 5

    out["age_38plus_flag"] = (out["age_ord"] >= 2).fillna(False).astype(int)
    out["age_40plus_flag"] = (out["age_ord"] >= 3).fillna(False).astype(int)
    out["age_43plus_flag"] = (out["age_ord"] >= 4).fillna(False).astype(int)
    out["age_unknown_flag"] = out["age_ord"].isna().astype(int)

    # 5) Count binning features.
    for col in ["수집된 신선 난자 수", "혼합된 난자 수", "해동 난자 수"]:
        if col in out.columns:
            out[f"{col}_bin"] = make_oocyte_bin(out[col])

    for col in ["총 생성 배아 수", "저장된 배아 수"]:
        if col in out.columns:
            out[f"{col}_bin"] = make_embryo_bin(out[col])

    if "이식된 배아 수" in out.columns:
        out["이식된 배아 수_bin"] = make_transfer_embryo_bin(out["이식된 배아 수"])

    # 6) Interaction features.
    egg_source_col = first_existing(out.columns, ["난자 출처", "egg_source"])
    treatment_col = first_existing(out.columns, ["시술 유형", "treatment_type"])
    reason_col = first_existing(out.columns, ["배아 생성 주요 이유", "embryo_creation_reason"])

    out["age_x_egg_source"] = age_text + "__" + safe_text_series(out, egg_source_col)
    out["treatment_x_specific_proc"] = safe_text_series(out, treatment_col) + "__" + specific_proc
    out["age_x_specific_proc"] = age_text + "__" + specific_proc
    out["reason_x_specific_proc"] = safe_text_series(out, reason_col) + "__" + specific_proc

    # 7) Ratio features.
    ratio_specs = {
        "transfer_per_created_embryo": ("이식된 배아 수", "총 생성 배아 수"),
        "stored_per_created_embryo": ("저장된 배아 수", "총 생성 배아 수"),
        "thawed_per_stored_embryo": ("해동된 배아 수", "저장된 배아 수"),
        "mixed_per_fresh_oocyte": ("혼합된 난자 수", "수집된 신선 난자 수"),
        "icsi_oocyte_rate": ("미세주입된 난자 수", "혼합된 난자 수"),
    }
    for new_col, (num_col, den_col) in ratio_specs.items():
        out[new_col] = safe_ratio(
            to_numeric_safe_series(out, num_col),
            to_numeric_safe_series(out, den_col),
        )

    return out


def fit_iqr_clip_bounds(df: pd.DataFrame) -> dict[str, tuple[float, float]]:
    outlier_candidate_cols = [
        "저장된 배아 수",
        "미세주입에서 생성된 배아 수",
        "미세주입된 난자 수",
        "혼합된 난자 수",
        "총 생성 배아 수",
        "수집된 신선 난자 수",
        "파트너 정자와 혼합된 난자 수",
        "임신 시도 또는 마지막 임신 경과 연수",
    ]
    bounds = {}
    for col in outlier_candidate_cols:
        if col not in df.columns:
            continue
        values = pd.to_numeric(df[col], errors="coerce")
        if values.notna().sum() == 0:
            continue
        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1
        if pd.isna(iqr):
            continue
        bounds[col] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    return bounds


def apply_iqr_clipping_features(df: pd.DataFrame, bounds: dict[str, tuple[float, float]]) -> pd.DataFrame:
    out = df.copy()
    for col, (lower_bound, upper_bound) in bounds.items():
        if col in out.columns:
            out[f"{col}_clipped"] = pd.to_numeric(out[col], errors="coerce").clip(lower_bound, upper_bound)
    return out


def normalize_binary_target(target: pd.Series) -> pd.Series:
    """Convert the target column to 0/1 while preserving unusable/missing labels as NaN."""
    numeric = pd.to_numeric(target, errors="coerce")
    non_missing_source = target.notna()

    if numeric[non_missing_source].notna().all():
        unique_values = sorted(pd.Series(numeric[non_missing_source]).dropna().unique())
        if set(unique_values).issubset({0, 1}):
            return numeric.astype(float)
        if len(unique_values) == 2:
            mapping = {unique_values[0]: 0.0, unique_values[1]: 1.0}
            print(f"Target values mapped to binary labels: {mapping}")
            return numeric.map(mapping).astype(float)

    positive_labels = {
        "1", "1.0", "true", "t", "yes", "y", "success", "succeeded",
        "pregnant", "pregnancy", "positive", "pos", "성공", "임신", "양성",
    }
    negative_labels = {
        "0", "0.0", "false", "f", "no", "n", "fail", "failed",
        "not pregnant", "negative", "neg", "실패", "미임신", "비임신", "음성",
    }

    def map_label(value: object) -> float:
        if pd.isna(value):
            return np.nan
        key = str(value).strip().lower()
        if key in positive_labels:
            return 1.0
        if key in negative_labels:
            return 0.0
        return np.nan

    return target.map(map_label).astype(float)


train_base = train.drop(columns=[target_col])
test_base = test.copy()

train_fe = add_missing_pattern_features(train_base, feature_columns)
test_fe = add_missing_pattern_features(test_base, feature_columns)

train_fe = add_v5_style_domain_features(train_fe)
test_fe = add_v5_style_domain_features(test_fe)

clip_bounds = fit_iqr_clip_bounds(train_fe)
train_fe = apply_iqr_clipping_features(train_fe, clip_bounds)
test_fe = apply_iqr_clipping_features(test_fe, clip_bounds)

if id_col is not None and id_col in train_fe.columns:
    train_fe = train_fe.drop(columns=[id_col])
if id_col is not None and id_col in test_fe.columns:
    test_ids = test_fe[id_col].copy()
    test_fe = test_fe.drop(columns=[id_col])
else:
    test_ids = None

X = train_fe.copy()
y_raw = train[target_col].copy()
y = normalize_binary_target(y_raw)

missing_target_mask = y.isna()
if missing_target_mask.any():
    dropped = int(missing_target_mask.sum())
    print(f"Dropping {dropped} rows with missing or unusable target labels before model training.")
    print("Raw target values among dropped rows:")
    print(y_raw[missing_target_mask].value_counts(dropna=False).head(10))

X = X.loc[~missing_target_mask].reset_index(drop=True)
y = y.loc[~missing_target_mask].astype(int).reset_index(drop=True)

if len(X) == 0:
    raise ValueError("No training rows remain after removing missing target labels.")
if y.isna().any():
    raise ValueError("Target still contains NaN after cleaning.")
if y.nunique(dropna=True) != 2:
    raise ValueError(f"Binary classification requires exactly 2 target classes. Found: {y.value_counts(dropna=False).to_dict()}")

X_test = test_fe.reindex(columns=X.columns, fill_value=np.nan).reset_index(drop=True)
if test_ids is not None:
    test_ids = test_ids.reset_index(drop=True)

added_feature_cols = [col for col in X.columns if col.startswith("f_") or col.endswith(("_flag", "_bin", "_clipped", "_is_missing"))]
ratio_cols = [
    "transfer_per_created_embryo",
    "stored_per_created_embryo",
    "thawed_per_stored_embryo",
    "mixed_per_fresh_oocyte",
    "icsi_oocyte_rate",
]
interaction_cols = [
    "age_x_egg_source",
    "treatment_x_specific_proc",
    "age_x_specific_proc",
    "reason_x_specific_proc",
]

print("X shape after target cleaning:", X.shape)
print("X_test shape:", X_test.shape)
print("Target distribution:")
print(y.value_counts().sort_index())
print("Feature count:", X.shape[1])
print("Added flag/bin/clipped/missing features:", len(added_feature_cols))
print("Ratio features present:", [col for col in ratio_cols if col in X.columns])
print("Interaction features present:", [col for col in interaction_cols if col in X.columns])
print("Clipping features present:", [f"{col}_clipped" for col in clip_bounds if f"{col}_clipped" in X.columns])


## 5. 전처리 파이프라인

범주형 변수는 빈도/순서의 임의성을 최소화하기 위해 `OrdinalEncoder`와 결측 대체를 결합한다. 수치형 변수는 중앙값 대체를 사용한다. 이 단계는 모델별 학습 fold 내부에서만 적합되어 OOF 검증 누수를 방지한다.


In [ ]:
def build_preprocessor(X_frame: pd.DataFrame) -> ColumnTransformer:
    categorical_cols = X_frame.select_dtypes(include=["object", "category", "bool", "string"]).columns.tolist()
    numeric_cols = [col for col in X_frame.columns if col not in categorical_cols]

    numeric_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ])
    categorical_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_cols),
            ("cat", categorical_pipe, categorical_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


preprocessor = build_preprocessor(X)
print(preprocessor)


## 6. OOF 검증과 모델 학습

OOF 검증은 제출 파일 생성 전 내부 일관성을 확인하기 위한 절차이다. IVF/ART 예측 모델은 데이터셋, 환자군, 시술 프로토콜에 따라 성능이 달라질 수 있으므로, 단일 점수보다 fold 간 변동성과 임상적으로 설명 가능한 변수 패턴을 함께 확인한다 (Liu et al., 2021; Fu et al., 2025).


In [ ]:
def optional_import_models() -> dict[str, object]:
    optional_models = {}

    try:
        from catboost import CatBoostClassifier
        optional_models["catboost"] = CatBoostClassifier
    except Exception as exc:
        print(f"CatBoost unavailable. Falling back without it. ({type(exc).__name__}: {exc})")

    try:
        from lightgbm import LGBMClassifier
        optional_models["lightgbm"] = LGBMClassifier
    except Exception as exc:
        print(f"LightGBM unavailable. Falling back without it. ({type(exc).__name__}: {exc})")

    try:
        from xgboost import XGBClassifier
        optional_models["xgboost"] = XGBClassifier
    except Exception as exc:
        print(f"XGBoost unavailable. Falling back without it. ({type(exc).__name__}: {exc})")

    return optional_models



def estimator_is_usable_in_pipeline(estimator: object) -> bool:
    """Return False for optional libraries that are installed but incompatible with the current sklearn version."""
    try:
        from sklearn.utils._tags import get_tags
        get_tags(estimator)
        return True
    except Exception as exc:
        print(
            f"Skipping {estimator.__class__.__name__}: incompatible with this sklearn environment "
            f"({type(exc).__name__}: {exc})"
        )
        return False


def make_models(seed: int) -> dict[str, Pipeline]:
    base_preprocessor = build_preprocessor(X)

    if CFG.run_mode == "smoke":
        boosting_estimators = 40
        tree_estimators = 60
        hist_max_iter = 20
        n_jobs = 1
    else:
        boosting_estimators = 700
        tree_estimators = 500
        hist_max_iter = 300
        n_jobs = -1

    optional_models = optional_import_models()
    models: dict[str, Pipeline] = {}

    if "catboost" in optional_models:
        CatBoostClassifier = optional_models["catboost"]
        cat_estimator = CatBoostClassifier(
            iterations=boosting_estimators,
            learning_rate=0.035,
            depth=6,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
        )
        if estimator_is_usable_in_pipeline(cat_estimator):
            models["catboost"] = Pipeline(steps=[
                ("preprocess", clone(base_preprocessor)),
                ("model", cat_estimator),
            ])

    if "lightgbm" in optional_models:
        LGBMClassifier = optional_models["lightgbm"]
        lgb_estimator = LGBMClassifier(
            n_estimators=boosting_estimators,
            learning_rate=0.035,
            num_leaves=31,
            subsample=0.85,
            colsample_bytree=0.85,
            random_state=seed,
            n_jobs=n_jobs,
            verbose=-1,
        )
        if estimator_is_usable_in_pipeline(lgb_estimator):
            models["lightgbm"] = Pipeline(steps=[
                ("preprocess", clone(base_preprocessor)),
                ("model", lgb_estimator),
            ])

    if "xgboost" in optional_models:
        XGBClassifier = optional_models["xgboost"]
        xgb_estimator = XGBClassifier(
            n_estimators=boosting_estimators,
            learning_rate=0.035,
            max_depth=4,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=seed,
            n_jobs=n_jobs,
        )
        if estimator_is_usable_in_pipeline(xgb_estimator):
            models["xgboost"] = Pipeline(steps=[
                ("preprocess", clone(base_preprocessor)),
                ("model", xgb_estimator),
            ])

    if not models:
        print("External boosting libraries were not found. Using sklearn fallback models.")
        models = {
            "hist_gradient_boosting": Pipeline(steps=[
                ("preprocess", clone(base_preprocessor)),
                ("model", HistGradientBoostingClassifier(
                    learning_rate=0.04,
                    max_iter=hist_max_iter,
                    l2_regularization=0.05,
                    random_state=seed,
                )),
            ]),
            "extra_trees": Pipeline(steps=[
                ("preprocess", clone(base_preprocessor)),
                ("model", ExtraTreesClassifier(
                    n_estimators=tree_estimators,
                    min_samples_leaf=3,
                    max_features="sqrt",
                    n_jobs=n_jobs,
                    random_state=seed,
                )),
            ]),
            "random_forest": Pipeline(steps=[
                ("preprocess", clone(base_preprocessor)),
                ("model", RandomForestClassifier(
                    n_estimators=tree_estimators,
                    min_samples_leaf=4,
                    max_features="sqrt",
                    n_jobs=n_jobs,
                    random_state=seed,
                )),
            ]),
        }

    print(
        f"Model settings: run_mode={CFG.run_mode}, "
        f"boosting_estimators={boosting_estimators}, tree_estimators={tree_estimators}, n_jobs={n_jobs}"
    )
    print("Models:", list(models.keys()))
    return models


def predict_positive_probability(model: Pipeline, X_frame: pd.DataFrame) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_frame)
        if proba.shape[1] == 2:
            return proba[:, 1]
        classes = getattr(model, "classes_", np.array([0]))
        only_class = classes[0] if len(classes) else 0
        return np.full(len(X_frame), float(only_class == 1), dtype=float)

    decision = model.decision_function(X_frame)
    return 1 / (1 + np.exp(-decision))


def get_effective_n_splits(y_true: pd.Series, requested_splits: int) -> int:
    class_counts = y_true.value_counts()
    min_class_count = int(class_counts.min())
    effective = min(int(requested_splits), min_class_count)

    if effective < 2:
        raise ValueError(
            "Not enough examples in the minority class for StratifiedKFold. "
            f"class_counts={class_counts.to_dict()}, requested_splits={requested_splits}"
        )

    if effective != requested_splits:
        print(
            f"Adjusted n_splits from {requested_splits} to {effective} "
            f"because the minority class has only {min_class_count} rows."
        )

    return effective


models = make_models(CFG.seed)
effective_n_splits = get_effective_n_splits(y, CFG.n_splits)
skf = StratifiedKFold(n_splits=effective_n_splits, shuffle=True, random_state=CFG.seed)

oof_bank = pd.DataFrame(index=X.index)
prediction_bank = pd.DataFrame(index=X_test.index)
cv_records = []

for model_name, model in models.items():
    oof_pred = np.full(len(X), np.nan, dtype=float)
    test_pred_folds = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        fold_model = clone(model)
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        if y_tr.isna().any() or y_va.isna().any():
            raise ValueError(f"NaN target detected inside fold {fold} for {model_name}.")

        fold_model.fit(X_tr, y_tr)
        va_pred = predict_positive_probability(fold_model, X_va)
        test_pred = predict_positive_probability(fold_model, X_test)

        oof_pred[va_idx] = va_pred
        test_pred_folds.append(test_pred)

        fold_auc = roc_auc_score(y_va, va_pred)
        cv_records.append({"model": model_name, "fold": fold, "auc": fold_auc})
        print(f"{model_name} fold {fold}: AUC={fold_auc:.5f}")

    if np.isnan(oof_pred).any():
        raise ValueError(f"OOF prediction contains NaN for model={model_name}.")

    oof_bank[model_name] = oof_pred
    prediction_bank[model_name] = np.mean(test_pred_folds, axis=0)
    model_auc = roc_auc_score(y, oof_pred)
    print(f"{model_name} OOF AUC: {model_auc:.5f}")

cv_summary = pd.DataFrame(cv_records)
cv_summary_path = CFG.output_dir / "cv_summary_v6_3.csv"
cv_summary.to_csv(cv_summary_path, index=False)
print(f"Saved CV summary: {cv_summary_path}")


## 7. Prediction Bank와 앙상블

각 모델의 test 예측값은 `prediction_bank`에 저장한다.  
OOF AUC가 높은 모델에 더 큰 비중을 주되, 특정 모델 하나에만 의존하지 않도록 가중 평균 앙상블을 사용한다. 성능 수치는 실행 환경과 데이터 버전에 따라 달라질 수 있으므로, 보고서에는 현재 실행 결과를 확인한 뒤 기재한다.


In [ ]:
def safe_roc_auc_score(y_true: pd.Series | np.ndarray, y_score: pd.Series | np.ndarray) -> float:
    y_true_series = pd.Series(y_true)
    if y_true_series.nunique(dropna=True) != 2:
        return np.nan
    return float(roc_auc_score(y_true_series, y_score))


def auc_weighted_ensemble(
    oof_frame: pd.DataFrame,
    pred_frame: pd.DataFrame,
    y_true: pd.Series,
) -> tuple[np.ndarray, np.ndarray, pd.Series]:
    if oof_frame.empty or pred_frame.empty:
        raise ValueError("OOF/prediction bank is empty. Run the model training cell first.")

    aucs = oof_frame.apply(lambda col: safe_roc_auc_score(y_true, col), axis=0)
    valid_aucs = aucs.fillna(0.5)
    centered = (valid_aucs - 0.5).clip(lower=0)

    if centered.sum() == 0:
        weights = pd.Series(1 / len(valid_aucs), index=valid_aucs.index)
    else:
        weights = centered / centered.sum()

    oof_ensemble = np.average(oof_frame.values, axis=1, weights=weights.values)
    test_ensemble = np.average(pred_frame.values, axis=1, weights=weights.values)
    return oof_ensemble, test_ensemble, weights


oof_ensemble, test_ensemble, ensemble_weights = auc_weighted_ensemble(oof_bank, prediction_bank, y)
ensemble_auc = safe_roc_auc_score(y, oof_ensemble)

prediction_bank["ensemble_v6_3"] = test_ensemble
oof_bank["ensemble_v6_3"] = oof_ensemble

prediction_bank_path = CFG.output_dir / "prediction_bank_v6_3.csv"
oof_bank_path = CFG.output_dir / "oof_bank_v6_3.csv"
weights_path = CFG.output_dir / "ensemble_weights_v6_3.csv"

prediction_bank.to_csv(prediction_bank_path, index=False)
oof_bank.to_csv(oof_bank_path, index=False)
ensemble_weights.rename("weight").to_csv(weights_path)

print("OOF ensemble AUC:", round(ensemble_auc, 6) if not np.isnan(ensemble_auc) else "nan")
print("Ensemble weights:")
print(ensemble_weights)
print(f"Saved prediction bank: {prediction_bank_path}")
print(f"Saved OOF bank: {oof_bank_path}")
print(f"Saved weights: {weights_path}")

## 8. 결과 해석 및 제출용 요약

### 8.1 모델 관점 해석

- 본 노트북은 OOF 기반 내부 검증 결과를 우선 확인한다. fold 간 차이가 크다면 평균 점수만으로 모델을 판단하지 않고, 데이터 분할에 따른 변동성도 함께 본다.
- CatBoost, LightGBM, XGBoost가 설치된 환경에서는 세 boosting 모델을 우선 사용한다. 해당 라이브러리가 없는 환경에서는 실행 검증을 위해 sklearn 기반 대체 모델을 사용한다.
- 앙상블은 각 모델의 예측을 `prediction_bank`에 저장한 뒤 OOF 성능을 기준으로 가중 평균한다. 이 방식은 단일 모델 하나에 제출 결과가 과도하게 의존하는 것을 줄이기 위한 절차이다.
- 최종 보고서에 기재하는 성능 수치와 feature 개수는 반드시 현재 실행 결과를 확인한 뒤 반영한다.

### 8.2 임상 변수 해석

- 여성 연령 및 난소 기능 관련 변수는 IVF/ICSI outcome을 설명할 때 반복적으로 언급되는 주요 축이다. 다만 본 모델에서는 원인 추정이 아니라 예측 신호로 해석한다 (Murugappan et al., 2023; Carson & Kallen, 2021).
- 배아 발달, 동결/해동, 이식 관련 변수는 치료 프로토콜과 착상 가능성 차이를 반영할 수 있다. `f_embryo_missing_count`, `f_embryo_all_missing`, `process_missing_count` 같은 결측 패턴 feature는 배아 정보의 부재 자체보다 시술 유형과 기록 구조 차이를 함께 나타내는 보조 신호로 해석한다 (Liu et al., 2021; Zhao et al., 2022).
- `fer_inferred_flag`, `icsi_inferred_flag`, `natural_cycle_inferred_flag`는 각 행에 이미 기록된 시술 정보를 고정 규칙으로 변환한 변수다. target을 이용해 만든 변수가 아니므로 leakage보다는 feature engineering으로 보는 것이 적절하다.
- ICSI 및 남성 요인 관련 변수는 수정 방식과 난임 원인 구성을 반영할 수 있으나, 대회 데이터만으로 개별 환자의 생물학적 원인을 단정하지 않는다 (Chen et al., 2022; Murugappan et al., 2023).

### 8.3 제출용 한계 문장

본 분석은 제공된 tabular 데이터만으로 임신 성공 여부를 예측한 결과이다.  
IVF/ART 결과는 환자 연령, 난소 예비력, 배아 상태, 자궁내막 환경, 시술 프로토콜 등 다양한 요인의 영향을 받으므로, 모델 예측은 실제 의학적 판단을 대체하기보다 데이터 기반 참고 정보로 해석해야 한다 (Carson & Kallen, 2021; Mohamed Salih et al., 2026).


## 9. Submission 생성

제출 파일은 `sample_submission.csv`의 컬럼 구조를 보존하고, 예측 대상 컬럼에 `ensemble_v6_3` 값을 채운다. 파일명은 `submission_team_report_v6_3_YYYYMMDD_HHMMSS.csv` 형식으로 저장한다.


In [ ]:
def infer_submission_target_column(sample: pd.DataFrame) -> str:
    non_id_columns = [col for col in sample.columns if col not in CFG.id_candidates]
    if len(non_id_columns) == 1:
        return non_id_columns[0]
    for candidate in CFG.target_candidates:
        if candidate in sample.columns:
            return candidate
    return sample.columns[-1]


def validate_submission_frame(
    frame: pd.DataFrame,
    sample: pd.DataFrame,
    target_column: str,
) -> None:
    if len(frame) != len(sample):
        raise ValueError(f"Submission row count mismatch: {len(frame)} != {len(sample)}")

    if list(frame.columns) != list(sample.columns):
        raise ValueError(
            "Submission columns must match sample_submission exactly. "
            f"submission={list(frame.columns)}, sample={list(sample.columns)}"
        )

    sample_id_col = find_first_existing(sample.columns, CFG.id_candidates)
    if sample_id_col is not None and not frame[sample_id_col].equals(sample[sample_id_col]):
        raise ValueError(f"Submission ID column order/value mismatch: {sample_id_col}")

    if frame[target_column].isna().any():
        raise ValueError(f"Submission target column contains NaN: {target_column}")

    if not frame[target_column].between(0, 1).all():
        bad = frame.loc[~frame[target_column].between(0, 1), target_column].head()
        raise ValueError(f"Submission probabilities must be within [0, 1]. Examples: {bad.tolist()}")


if len(test_ensemble) != len(sample_submission):
    raise ValueError(
        f"Prediction length mismatch: len(test_ensemble)={len(test_ensemble)} "
        f"but len(sample_submission)={len(sample_submission)}"
    )

submission = sample_submission.copy()
submission_target_col = infer_submission_target_column(submission)
submission[submission_target_col] = np.clip(test_ensemble, 0, 1)

# Keep the sample_submission ID/order as the contract. Only overwrite when
# the same ID column exists in both test and sample and the lengths match.
if id_col is not None and id_col in submission.columns and test_ids is not None:
    if len(test_ids) != len(submission):
        raise ValueError(f"test ID length mismatch: {len(test_ids)} != {len(submission)}")
    if submission[id_col].equals(test_ids):
        submission[id_col] = test_ids.values
    else:
        print(
            f"Warning: test IDs and sample_submission IDs differ for column {id_col}. "
            "Keeping sample_submission ID order."
        )

validate_submission_frame(submission, sample_submission, submission_target_col)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
submission_path = CFG.output_dir / f"{CFG.submission_prefix}_{timestamp}.csv"
submission.to_csv(submission_path, index=False)

# Re-read once to catch write/read formatting issues.
saved_submission = pd.read_csv(submission_path)
validate_submission_frame(saved_submission, sample_submission, submission_target_col)

print("submission target column:", submission_target_col)
print("submission path:", submission_path)
print("submission shape:", submission.shape)
submission.head()

## 10. 참고문헌

노트북 본문에는 필요한 위치에만 간결한 `(저자 또는 기관, 연도)` 형식으로 표기하였다.

- Carson, S. A., & Kallen, A. N. (2021). Diagnosis and Management of Infertility: A Review.
- Chen, H. H., et al. (2022). Clinical predictors of embryo quality among women of advanced age receiving ICSI cycles.
- Fu, X., et al. (2025). Enhancing frozen-thawed embryo transfer outcomes and treatment personalization through machine learning models.
- Liu, Y., et al. (2021). Multifactor prediction of embryo transfer outcomes based on a machine learning algorithm.
- Mohamed Salih, E. L., et al. (2026). Clinical predictors of successful pregnancy after IVF: A comprehensive systematic review of evidence.
- Murugappan, G., et al. (2023). IVF and ICSI predictive factors: female age, ovarian reserve, male age, and male factor.
- Zhao, J., et al. (2022). Nomogram prediction for clinical pregnancy in freeze-thawed embryo transfer.
